# Sprint 10 · Gate sobre predições de validação · colab

Antes de executar: leia `sprints/README.md`. Input obrigatório: sprint10_val.csv, com predições reais de validação (veja o schema no README). Não usa o teste final. Não cole tokens no notebook.

In [ ]:
from pathlib import Path
import subprocess, sys
from zipfile import ZipFile

from google.colab import drive
drive.mount('/content/drive')
base = Path('/content/drive/MyDrive/mammo')
SOURCE_ZIP = base / 'mammo-codigo.zip'
PROJECT = Path('/content/mammo-codigo')
DATA = base / 'desenvolvimento'
OUTPUT = base / 'resultados' 
# Altere DATA se o input estiver em outro caminho. Não aponte para o teste final.
SMOKE = True  # S7–S9: uma época, sem pesos externos. False usa o protocolo configurado.
ARM = 'batch-16-fixed'  # somente S9; um braço por execução

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f'Anexe o ZIP de código em {SOURCE_ZIP}; veja sprints/README.md')
with ZipFile(SOURCE_ZIP) as z:
    for item in z.infolist():
        dest = (PROJECT / item.filename).resolve()
        if not dest.is_relative_to(PROJECT.resolve()):
            raise ValueError('Arquivo de código com caminho inválido')
    z.extractall(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))
if not (PROJECT / 'configs' / 'sprints.toml').is_file():
    raise RuntimeError('ZIP incorreto: gere com python scripts/package_sprints.py')


In [ ]:
from datetime import datetime, timezone
from zipfile import ZIP_DEFLATED, ZipFile


def pack_results():
    """Compacta toda a saída; roda mesmo se o experimento falhar."""
    archive = OUTPUT / 'sprint-10-resultados.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as stream:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path != archive:
                stream.write(path, path.relative_to(OUTPUT))
    assert ZipFile(archive).testzip() is None, 'ZIP corrompido'
    return archive


try:
    from project_cnn_mammo.protocols import run_sprint
    out = OUTPUT / 'sprint-10' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    print(run_sprint(10, DATA, out, smoke=False, arm=ARM, manifest_path=globals().get('MANIFEST')))
    print('Saída:', out)
finally:
    archive = pack_results()
    print('ZIP:', archive, archive.stat().st_size, 'bytes')
    from google.colab import files
    files.download(archive)  # baixa o ZIP automaticamente no navegador